[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20I%20-%20The%20Port%20%26%20Container%20Terminal%20%28Problems%201-46%29/A.%20Foundations%20of%20Terminal%20Operations%20%28The%20Building%20Blocks%29/01.%20The%20Single%20Crane%20Lift%20Sequence%20Problem/P1-Tier-1_executed.ipynb)

---


# 1. The Single Crane Lift Sequence Problem

## Tier 1 — Dynamic Programming

This notebook develops and implements a dynamic programming (DP) solution for sequencing single and dual container lifts. The crane must lift every container exactly once. The plan determines both the order of the lifts and whether each action uses single or dual mode.

### Learning goals

- Explain what a DP state records and why it must include the remaining containers, crane position, and current mode.
- Formulate the minimum-time sequencing problem as a recursive optimization model.
- Implement the recurrence with memoization and reconstruct an optimal sequence of actions.
- Validate the reconstructed plan and interpret its time components and visualizations.

### Inputs and outputs

The input is a six-container instance with container positions and weights, a bay layout, crane starting conditions, time parameters, and dual-lift feasibility rules. The notebook returns a minimum-time lift plan, a step-by-step time table, and visualizations of the operations.

### Scope and modeling assumptions

The bay is a 2 × 3 grid numbered in serpentine order. Grid adjacency is used to decide whether two containers may be lifted together. The travel model is a separate one-dimensional approximation: the crane's position is represented by the scalar position index, and travel time is proportional to the absolute difference between scalar positions. A dual lift's scalar target is the mean of its two position indices. Consequently, the bay plot shows the geometric locations and sequence of visits, while the travel-time calculation follows the scalar index model; the plotted arrow lengths are not a measurement of modeled travel time. The model does not include congestion, variable speeds, uncertainty, or constraints beyond those stated below.

### Why the instance is small

The recursion considers many subsets of containers. A six-container instance keeps the state transitions small enough to inspect while still demonstrating single lifts, dual lifts, setup times, and plan reconstruction.


In [ ]:
# Import the libraries used for data handling, numerical checks, and plots.
try:
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
except ImportError as e:
    raise ImportError(
        "Missing dependency. Install numpy, pandas, and matplotlib "
        "in the Python environment used by this notebook."
    ) from e

print("Dependencies imported successfully.")


## Concrete instance (6 containers)

The instance contains six containers in a 2 × 3 bay. Each container has an identifier, a position index, and a weight in tons.

### Time model

- A single lift takes 2.5 minutes of handling time.
- A dual lift takes 3.8 minutes of handling time.
- Switching between single and dual mode takes 1.2 minutes. No setup time is charged when the mode stays the same.
- Travel time is `0.8 × |position_from − position_to|` minutes.

These are additive time components: each action incurs travel time, any required setup time, and the handling time for its lift mode.

### Dual-lift feasibility

A pair can be lifted together only if its containers are adjacent in the 2D grid and their combined weight does not exceed the dual weight limit. Both conditions are necessary. The grid layout and the scalar travel approximation serve different purposes: adjacency uses grid coordinates, whereas travel uses position indices.


In [ ]:
from dataclasses import dataclass
from functools import lru_cache
from itertools import combinations
from typing import Any, Dict, FrozenSet, List, Literal


Mode = Literal["single", "dual"]


@dataclass(frozen=True)
class Container:
    """Container data needed by the Tier 1 model."""

    id: int
    pos: int
    weight: float


# The position indices follow the bay's serpentine numbering.
containers = [
    Container(1, 1, 8.5),
    Container(2, 2, 12.3),
    Container(3, 3, 15.7),
    Container(4, 4, 9.8),
    Container(5, 5, 11.2),
    Container(6, 6, 14.1),
]

# This lookup lets the DP retrieve a container's data from its identifier.
id_to_container = {container.id: container for container in containers}


# Time and feasibility parameters; times are measured in minutes.
HANDLE_SINGLE = 2.5
HANDLE_DUAL = 3.8
SETUP_TIME = 1.2
DUAL_WEIGHT_LIMIT = 20.0
TRAVEL_PER_UNIT = 0.8

START_POS = 1.0
START_MODE: Mode = "single"


def travel_time(pos_from: float, pos_to: float) -> float:
    """Return travel time under the scalar position-index approximation."""
    return TRAVEL_PER_UNIT * abs(pos_from - pos_to)


def dual_center_pos(id1: int, id2: int) -> float:
    """Return the scalar travel-model target for a dual lift."""
    pos1 = id_to_container[id1].pos
    pos2 = id_to_container[id2].pos
    return (pos1 + pos2) / 2.0


def dual_feasible(id1: int, id2: int) -> bool:
    """Check whether the combined weight is within the current limit."""
    weight1 = id_to_container[id1].weight
    weight2 = id_to_container[id2].weight
    return weight1 + weight2 <= DUAL_WEIGHT_LIMIT


# This list applies the weight condition; grid adjacency is applied in Step 1.
feasible_dual_pairs = [
    pair
    for pair in combinations([container.id for container in containers], 2)
    if dual_feasible(*pair)
]

feasible_dual_pairs


## Step 1 — Define when a dual lift is allowed

A dual lift removes two containers in one action, so it is allowed only when the pair satisfies both the weight limit and the spatial adjacency rule. The bay uses this serpentine numbering:

```text
Row 0:  1  2  3
Row 1:  6  5  4
```

For positions with grid coordinates `(r_i, c_i)` and `(r_j, c_j)`, Manhattan distance is `|r_i − r_j| + |c_i − c_j|`. The positions are adjacent exactly when this distance equals 1. This grid rule is independent of the scalar position-index travel approximation. For example, positions 2 and 5 share a grid edge even though their indices are not consecutive.


In [ ]:
# Grid coordinates encode the physical 2D layout used by the adjacency rule.
pos_to_rc = {
    1: (0, 0),
    2: (0, 1),
    3: (0, 2),
    4: (1, 2),
    5: (1, 1),
    6: (1, 0),
}


def are_adjacent_on_grid(pos_i: int, pos_j: int) -> bool:
    """Return whether two grid positions have Manhattan distance one."""
    row_i, col_i = pos_to_rc[pos_i]
    row_j, col_j = pos_to_rc[pos_j]
    return abs(row_i - row_j) + abs(col_i - col_j) == 1


# Keep pairs that satisfy both the weight condition and grid adjacency.
adjacent_dual_pairs = []
for id1, id2 in feasible_dual_pairs:
    pos1 = id_to_container[id1].pos
    pos2 = id_to_container[id2].pos
    if are_adjacent_on_grid(pos1, pos2):
        adjacent_dual_pairs.append((id1, id2))

adjacent_dual_pairs


## Step 2 — Dynamic programming to find the best lift plan

### Intuition: what a state records

A DP state summarizes everything from the past that can affect the best future decision. Here, `remaining` records which containers have not yet been lifted, `pos` records the crane's current scalar position, and `mode` records the mode currently set on the crane. The order of actions that led to this state does not need to be stored: once these three values are known, the future choices and their costs are determined by the model.

The mode is essential because the next action may require a setup time. The position is essential because it determines travel time. The remaining set is essential because it determines which lifts are still possible. Memoization stores the value for each encountered state so that different action sequences reaching the same state do not require solving its future repeatedly.

### Formal state and action sets

Let `N` be the set of container identifiers. A state is `(R, x, m)`, where `R ⊆ N` is the set of remaining containers, `x` is the current scalar position, and `m ∈ {single, dual}` is the current mode. At a state, a single action selects any `i ∈ R`. A dual action selects an adjacent, weight-feasible pair `{i,j} ⊆ R`.

For an action with mode `q` and target position `y`, define its step cost as

$$
c(x,m;q,y)
= 0.8|x-y|
+ 1.2\,\mathbf{1}[m \ne q]
+ h(q),
\qquad
h(q)=
\begin{cases}
2.5, & q=\text{single},\\
3.8, & q=\text{dual}.
\end{cases}
$$

Here, `1[m ≠ q]` is 1 when the mode changes and 0 otherwise. The code implements this cost in `lift_cost_components` and `lift_cost`.

### Bellman recurrence

Let `V(R,x,m)` be the minimum additional time required to lift every container in `R`. The base case and transition are

$$
V(\varnothing,x,m)=0,
$$

$$
\begin{aligned}
V(R,x,m)=\min\Bigg\{\;
&\min_{i\in R}\left[c(x,m;\text{single},p_i)
+V(R\setminus\{i\},p_i,\text{single})\right],\\
&\min_{\substack{\{i,j\}\subseteq R:\\
\text{adjacent}(i,j),\\
w_i+w_j\le W}}
\left[c(x,m;\text{dual},\tfrac{p_i+p_j}{2})
+V(R\setminus\{i,j\},\tfrac{p_i+p_j}{2},\text{dual})\right]
\;\Bigg\}.
\end{aligned}
$$

In this recurrence, `p_i` and `w_i` are container `i`'s position and weight, `W` is `DUAL_WEIGHT_LIMIT`, and `adjacent(i,j)` is the grid rule from Step 1. A minimum over an empty set of dual actions contributes no candidate. The base case says that no additional time is needed when no containers remain.

The implementation is the cached function `dp(remaining, pos, mode)`. Each recursive action removes one or two containers, so the recursion always progresses toward the base case. The number of subsets can grow exponentially with the number of containers; memoization avoids recomputing a state but does not remove that exponential growth. The second part of this step, `reconstruct`, follows actions whose immediate cost plus future DP value attains the state value, recovering an optimal plan rather than only its objective value.


In [ ]:
def lift_cost_components(
    current_pos: float,
    current_mode: Mode,
    lift_mode: Mode,
    target_pos: float,
) -> Dict[str, float]:
    """Return the travel, setup, and handling times for one lift action."""
    setup = SETUP_TIME if current_mode != lift_mode else 0.0
    handling = HANDLE_DUAL if lift_mode == "dual" else HANDLE_SINGLE
    travel = travel_time(current_pos, target_pos)
    return {"travel_time": travel, "setup_time": setup, "handling_time": handling}


def lift_cost(
    current_pos: float,
    current_mode: Mode,
    lift_mode: Mode,
    target_pos: float,
) -> float:
    """Return the total time for one action under the model."""
    components = lift_cost_components(
        current_pos, current_mode, lift_mode, target_pos
    )
    return sum(components.values())


@lru_cache(maxsize=None)
def dp(remaining: FrozenSet[int], pos: float, mode: Mode) -> float:
    """Return the minimum additional time needed to lift remaining containers."""
    if not remaining:
        return 0.0

    best = float("inf")

    # Every remaining container is a possible next single-lift action.
    for cid in sorted(remaining):
        container = id_to_container[cid]
        target_pos = float(container.pos)
        step = lift_cost(pos, mode, "single", target_pos)
        remaining_after = frozenset(remaining - {cid})
        candidate = step + dp(remaining_after, target_pos, "single")
        best = min(best, candidate)

    # A dual action is considered only when both containers remain and their pair is feasible.
    for id1, id2 in combinations(sorted(remaining), 2):
        if (id1, id2) not in adjacent_dual_pairs:
            continue

        target_pos = dual_center_pos(id1, id2)
        step = lift_cost(pos, mode, "dual", target_pos)
        remaining_after = frozenset(remaining - {id1, id2})
        candidate = step + dp(remaining_after, target_pos, "dual")
        best = min(best, candidate)

    return best


def reconstruct() -> List[Dict[str, Any]]:
    """Recover an optimal sequence by following DP-optimal actions."""
    remaining: FrozenSet[int] = frozenset(container.id for container in containers)
    pos = float(START_POS)
    mode: Mode = START_MODE
    plan: List[Dict[str, Any]] = []

    while remaining:
        best_choice = None
        best_value = float("inf")

        # Examine single actions in identifier order for reproducible tie handling.
        for cid in sorted(remaining):
            container = id_to_container[cid]
            target_pos = float(container.pos)
            remaining_after = frozenset(remaining - {cid})
            value = lift_cost(pos, mode, "single", target_pos)
            value += dp(remaining_after, target_pos, "single")
            if value < best_value:
                best_value = value
                best_choice = ("single", [cid], target_pos)

        # Examine feasible dual actions; sorted pairs make tie handling reproducible.
        for id1, id2 in combinations(sorted(remaining), 2):
            if (id1, id2) not in adjacent_dual_pairs:
                continue

            target_pos = dual_center_pos(id1, id2)
            remaining_after = frozenset(remaining - {id1, id2})
            value = lift_cost(pos, mode, "dual", target_pos)
            value += dp(remaining_after, target_pos, "dual")
            if value < best_value:
                best_value = value
                best_choice = ("dual", [id1, id2], target_pos)

        assert best_choice is not None, "At least one single action must be available."

        lift_mode, ids, target_pos = best_choice
        components = lift_cost_components(pos, mode, lift_mode, target_pos)
        step_time = sum(components.values())
        plan.append(
            {
                "mode": lift_mode,
                "containers": ids,
                "to_pos": target_pos,
                **components,
                "step_time": step_time,
            }
        )

        remaining = frozenset(remaining - set(ids))
        pos = target_pos
        mode = lift_mode

    return plan


# Solve from the specified initial crane position and mode.
opt_time = dp(
    frozenset(container.id for container in containers),
    float(START_POS),
    START_MODE,
)
plan = reconstruct()

opt_time, plan


## Step 3 — Inspect and validate the optimal plan

The DP value is the minimum total time, while the reconstructed plan lists the actions that attain it. Validation checks connect these two representations: every container must appear exactly once, each action must satisfy the model's feasibility rules, and the sum of the recorded time components must equal the DP objective within floating-point tolerance.

The table includes travel, setup, and handling contributions so that each step's total can be traced to the cost model. A rounded display is convenient for reading, but the validation uses the unrounded plan values.


In [ ]:
from math import isclose


# Preserve the unrounded plan values for validation and create a readable table.
df = pd.DataFrame(plan)
df.index = np.arange(1, len(df) + 1)
df.index.name = "step"
df["containers"] = df["containers"].apply(lambda ids: ", ".join(map(str, ids)))

component_columns = ["travel_time", "setup_time", "handling_time", "step_time"]
df[component_columns] = df[component_columns].round(3)

lifted_ids = [cid for step in plan for cid in step["containers"]]
all_lifted_exactly_once = sorted(lifted_ids) == sorted(
    container.id for container in containers
) and len(lifted_ids) == len(set(lifted_ids))
plan_total = sum(step["step_time"] for step in plan)
components_match = all(
    isclose(
        step["step_time"],
        step["travel_time"] + step["setup_time"] + step["handling_time"],
        rel_tol=1e-9,
        abs_tol=1e-9,
    )
    for step in plan
)

assert all_lifted_exactly_once, "The plan must lift every container exactly once."
assert components_match, "Each step time must equal the sum of its components."
assert isclose(plan_total, opt_time, rel_tol=1e-9, abs_tol=1e-9), (
    "The reconstructed plan total must match the DP objective."
)

print(f"DP minimum total time: {opt_time:.3f} min")
print(f"Reconstructed plan total: {plan_total:.3f} min")
print("Lifted container IDs:", sorted(lifted_ids))
print("Every container is lifted exactly once:", all_lifted_exactly_once)
print("Step costs equal their time components:", components_match)

df


## Additional operational visualizations

The time breakdown separates travel, setup, and handling contributions. Cumulative completion time shows how the objective accumulates across the sequence, and the timeline locates each operation on that cumulative time axis. Colors identify the lift mode; the legend makes that encoding explicit. These charts summarize the reconstructed plan and do not perform additional optimization.


In [ ]:
vis = pd.DataFrame(plan).copy()
vis["step"] = np.arange(1, len(vis) + 1)
vis["cum_time"] = vis["step_time"].cumsum()
vis["start_time"] = vis["cum_time"] - vis["step_time"]

color_map = {"single": "#2E90FA", "dual": "#12B76A"}
colors = [color_map[mode] for mode in vis["mode"]]

fig, axes = plt.subplots(1, 3, figsize=(16, 3.8))

# Stacked bars show how the modeled time for each action is composed.
bottom = np.zeros(len(vis))
for column, label, color in [
    ("travel_time", "Travel", "#6172F3"),
    ("setup_time", "Setup", "#F79009"),
    ("handling_time", "Handling", "#12B76A"),
]:
    axes[0].bar(
        vis["step"],
        vis[column],
        bottom=bottom,
        label=label,
        color=color,
        edgecolor="#FFFFFF",
    )
    bottom += vis[column].to_numpy()
axes[0].set_title("Time components by step")
axes[0].set_xlabel("Step")
axes[0].set_ylabel("Time (minutes)")
axes[0].legend(title="Component")
axes[0].grid(True, axis="y", alpha=0.25)

axes[1].plot(vis["step"], vis["cum_time"], marker="o", linewidth=2, color="#344054")
axes[1].set_title("Cumulative completion time")
axes[1].set_xlabel("Step")
axes[1].set_ylabel("Time (minutes)")
axes[1].grid(True, alpha=0.25)

axes[2].barh(
    y=vis["step"],
    width=vis["step_time"],
    left=vis["start_time"],
    color=colors,
    edgecolor="#101828",
    alpha=0.85,
)
axes[2].set_title("Operation timeline")
axes[2].set_xlabel("Time (minutes)")
axes[2].set_ylabel("Step")
axes[2].grid(True, axis="x", alpha=0.25)

from matplotlib.patches import Patch

mode_handles = [
    Patch(facecolor=color_map["single"], edgecolor="#101828", label="Single lift"),
    Patch(facecolor=color_map["dual"], edgecolor="#101828", label="Dual lift"),
]
axes[2].legend(handles=mode_handles, title="Lift mode", loc="best")

plt.tight_layout()
plt.show()


## Step 4 — Visualize and interpret the plan

The bay plot locates the containers and indicates the order in which the crane serves the selected targets. A dual action is drawn at the geometric midpoint of its two grid cells. The arrows communicate sequence and target locations; their lengths are not used to calculate travel time. The model's travel cost remains the scalar index difference defined earlier.

Use the plot together with the time table: a geometrically long-looking arrow need not have a proportionally large modeled travel cost. A dual action handles two containers together, but its total cost can also include travel and a mode-switch setup. The DP balances these costs over the full remaining sequence rather than choosing each next action independently.


In [ ]:
# Plot grid geometry separately from the scalar travel-time calculation.
pos_to_xy = {position: (col, -row) for position, (row, col) in pos_to_rc.items()}

fig, ax = plt.subplots(figsize=(6, 3.8))

for position, (x, y) in pos_to_xy.items():
    ax.scatter(
        [x], [y], s=800, marker="s", facecolor="#F2F4F7", edgecolor="#344054"
    )
    ax.text(
        x, y, str(position), ha="center", va="center", fontsize=12, color="#101828"
    )

for container in containers:
    x, y = pos_to_xy[container.pos]
    ax.text(
        x,
        y - 0.25,
        f"ID {container.id}\n{container.weight:.1f} t",
        ha="center",
        va="top",
        fontsize=9,
        color="#475467",
    )

# The start marker and visit targets show spatial sequence, not travel-time scale.
start_position = int(round(START_POS))
start_x, start_y = pos_to_xy[start_position]
path_xy = [(start_x, start_y + 0.35)]

for step in plan:
    ids = step["containers"]
    if step["mode"] == "single":
        target_position = id_to_container[ids[0]].pos
        x, y = pos_to_xy[target_position]
    else:
        position1 = id_to_container[ids[0]].pos
        position2 = id_to_container[ids[1]].pos
        x1, y1 = pos_to_xy[position1]
        x2, y2 = pos_to_xy[position2]
        x, y = (x1 + x2) / 2.0, (y1 + y2) / 2.0
    path_xy.append((x, y + 0.35))

for (x1, y1), (x2, y2) in zip(path_xy[:-1], path_xy[1:]):
    ax.annotate(
        "",
        xy=(x2, y2),
        xytext=(x1, y1),
        arrowprops={"arrowstyle": "->", "linewidth": 2, "color": "#2E90FA"},
    )

ax.set_title("Bay layout and crane visit sequence")
ax.set_xlabel("Grid column")
ax.set_ylabel("Grid row (displayed top to bottom)")
ax.set_xticks([])
ax.set_yticks([])
ax.set_xlim(-0.75, 2.75)
ax.set_ylim(-1.75, 0.75)
ax.set_aspect("equal")
plt.tight_layout()
plt.show()


## Common pitfalls and debugging

- If no dual actions are available, inspect both conditions: the combined weight must be within the limit, and the positions must share a grid edge.
- If a dual action is allowed by the grid but seems distant by index, remember that grid adjacency and scalar travel distance are separate parts of this model.
- If a total time seems inconsistent, compare its travel, setup, and handling components with the action's current position and mode.
- For a dual action, the scalar crane position after the lift is the mean of the two position indices, as used by `dual_center_pos`.

## Next steps

The sensitivity analysis below varies the dual weight limit while keeping the instance, bay, starting conditions, and time parameters fixed. Larger instances can also be studied, but the number of subsets considered by the DP grows exponentially, so computational effort can increase rapidly. Further modeling work could add operational constraints while retaining the same DP approach when the resulting state and transitions remain well-defined.


## Sensitivity analysis: when do dual lifts appear?

A dual action appears in the optimization only if a pair is both adjacent and within the weight limit. Even when feasible pairs exist, the minimum-time plan may still prefer single lifts because the total sequence cost also includes travel, setup, and handling.

The experiment below keeps the container data, bay layout, starting conditions, and time parameters fixed, and varies only `DUAL_WEIGHT_LIMIT`. For each limit, it recomputes the weight-feasible adjacent pairs, clears the memoized values that depend on that pair set, and solves the same DP recurrence. The reported number of dual steps is a property of the selected optimal plan; if multiple optimal plans exist, the reconstruction's deterministic action order selects one of them.


In [ ]:
def solve_with_dual_limit(limit: float) -> Dict[str, Any]:
    """Solve the same DP model for a temporary dual weight limit."""
    global DUAL_WEIGHT_LIMIT, feasible_dual_pairs, adjacent_dual_pairs

    old_limit = DUAL_WEIGHT_LIMIT
    old_feasible_pairs = feasible_dual_pairs.copy()
    old_adjacent_pairs = adjacent_dual_pairs.copy()

    try:
        DUAL_WEIGHT_LIMIT = float(limit)
        feasible_dual_pairs = [
            pair
            for pair in combinations([container.id for container in containers], 2)
            if dual_feasible(*pair)
        ]

        adjacent_dual_pairs = []
        for id1, id2 in feasible_dual_pairs:
            position1 = id_to_container[id1].pos
            position2 = id_to_container[id2].pos
            if are_adjacent_on_grid(position1, position2):
                adjacent_dual_pairs.append((id1, id2))

        # Cached DP values depend on the available dual-action set.
        dp.cache_clear()
        experiment_time = dp(
            frozenset(container.id for container in containers),
            float(START_POS),
            START_MODE,
        )
        experiment_plan = reconstruct()
        dual_steps = sum(step["mode"] == "dual" for step in experiment_plan)

        return {
            "dual_weight_limit": float(limit),
            "num_adjacent_dual_pairs": len(adjacent_dual_pairs),
            "adjacent_dual_pairs": adjacent_dual_pairs.copy(),
            "opt_time": float(experiment_time),
            "dual_steps": int(dual_steps),
        }
    finally:
        # Restore the main instance even if solving the experiment raises an error.
        DUAL_WEIGHT_LIMIT = old_limit
        feasible_dual_pairs = old_feasible_pairs
        adjacent_dual_pairs = old_adjacent_pairs
        dp.cache_clear()


limits_to_try: List[float] = [20.0, 22.0, 24.0, 26.0, 30.0]
rows = [solve_with_dual_limit(limit) for limit in limits_to_try]

pd.DataFrame(rows)
